In [1]:
import pandas as pd
import numpy as np
import textwrap
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('bbc_text_cls.csv')

In [3]:
df.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [4]:
doc = df[df.labels == 'business']['text'].sample(random_state = 42)

In [5]:
def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

In [6]:
sents = nltk.sent_tokenize(doc.iloc[0].split("\n", 1)[1])

In [7]:
featurizer = TfidfVectorizer(
    stop_words=stopwords.words('english'),
    norm='l1',
)

In [8]:
X = featurizer.fit_transform(sents)

In [9]:
# compute similarity matrix
S = cosine_similarity(X)

S.shape

(17, 17)

In [10]:
len(sents)

17

In [11]:
# normalize similarity matrix
S /= S.sum(axis=1, keepdims=True)

S[0].sum()

np.float64(1.0)

In [12]:
# uniform transition matrix
U = np.ones_like(S) / len(S)

U[0].sum()

# smoothed similarity matrix
factor = 0.15
S = (1 - factor) * S + factor * U

S[0].sum()

np.float64(0.9999999999999999)

In [13]:
# find the limiting / stationary distribution
eigenvals, eigenvecs = np.linalg.eig(S.T)

eigenvals

eigenvecs[:, 0]

eigenvecs[:, 0].dot(S)

eigenvecs[:, 0] / eigenvecs[:, 0].sum()

array([0.05907327-0.j, 0.06601563-0.j, 0.05402535-0.j, 0.06982824-0.j,
       0.06117038-0.j, 0.06099047-0.j, 0.06823848-0.j, 0.05250595-0.j,
       0.05433915-0.j, 0.05550753-0.j, 0.05025022-0.j, 0.05114976-0.j,
       0.05741304-0.j, 0.05906657-0.j, 0.05774684-0.j, 0.07175905-0.j,
       0.05092007-0.j])

In [14]:
limiting_dist = np.ones(len(S)) / len(S)
threshold = 1e-8
delta = float('inf')
iters = 0

while delta > threshold:
    iters += 1

    # Markov transition
    p = limiting_dist.dot(S)

    # compute change in limiting distribution
    delta = np.abs(p - limiting_dist).sum()

    # update limiting distribution
    limiting_dist = p
print(iters)

41


In [15]:
limiting_dist

limiting_dist.sum()

np.abs(
    eigenvecs[:, 0] / eigenvecs[:, 0].sum()
    - limiting_dist
).sum()

scores = limiting_dist

sort_idx = np.argsort(-scores)

In [16]:
# Many options for how to choose which sentences to include:

# 1) top N sentences
# 2) top N words
# 3) top X% sentences or top X% words
# 4) sentences with scores > average score
# 5) sentences with scores > factor * average score

# You also don't have to sort. May make more sense in order.

print("Generated summary:")
for i in sort_idx[:5]:
  print(wrap("%.2f: %s" % (scores[i], sents[i])))

Generated summary:
0.07: "The retail sales figures are very weak, but as Bank of England
governor Mervyn King indicated last night, you don't really get an
accurate impression of Christmas trading until about Easter," said Mr
Shaw.
0.07: A number of retailers have already reported poor figures for
December.
0.07: The ONS echoed an earlier caution from Bank of England governor
Mervyn King not to read too much into the poor December figures.
0.07: Retail sales dropped by 1% on the month in December, after a
0.6% rise in November, the Office for National Statistics (ONS) said.
0.06: Clothing retailers and non-specialist stores were the worst hit
with only internet retailers showing any significant growth, according
to the ONS.


In [17]:
doc.iloc[0].split("\n", 1)[0]

'Christmas sales worst since 1981'

In [18]:
def summarize(text, factor = 0.15):
  # extract sentences
  sents = nltk.sent_tokenize(text)
  # perform tf-idf
  featurizer = TfidfVectorizer(
  stop_words=stopwords.words('english'),
  norm='l1'
  )
  X = featurizer.fit_transform(sents)
  # compute similarity matrix
  S = cosine_similarity(X)
  # normalize similarity matrix
  S /= S.sum(axis=1, keepdims=True)
  # uniform transition matrix
  U = np.ones_like(S) / len(S)
  # smoothed similarity matrix
  S = (1 - factor) * S + factor * U
  # find the limiting / stationary distribution
  eigenvals, eigenvecs = np.linalg.eig(S.T)
  # compute scores
  scores = eigenvecs[:, 0] / eigenvecs[:, 0].sum()
  # sort the scores
  sort_idx = np.argsort(-scores)
  # print summary
  for i in sort_idx[:5]:
    print(wrap("%.2f: %s" % (scores[i], sents[i])))

In [19]:
doc = df[df.labels == 'entertainment']['text'].sample(random_state=123)
summarize(doc.iloc[0].split("\n", 1)[1])

0.11: Goodrem, Green Day and the Black Eyed Peas took home two awards
each.
0.10: As well as best female, Goodrem also took home the Pepsi Viewers
Choice Award, whilst Green Day bagged the prize for best rock video
for American Idiot.
0.10: Other winners included Green Day, voted best group, and the
Black Eyed Peas.
0.10: The Black Eyed Peas won awards for best R 'n' B video and
sexiest video, both for Hey Mama.
0.10: Local singer and songwriter Missy Higgins took the title of
breakthrough artist of the year, with Australian Idol winner Guy
Sebastian taking the honours for best pop video.


C:\Users\Anupam Tiwari\AppData\Local\Temp\ipykernel_21512\1608990715.py:26: ComplexWarning: Casting complex values to real discards the imaginary part
  print(wrap("%.2f: %s" % (scores[i], sents[i])))


In [20]:
doc.iloc[0].split("\n", 1)[0]

'Goodrem wins top female MTV prize'

## Libraries for Text Summarization

In [21]:
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer

In [22]:
summarizer = TextRankSummarizer()

parser = PlaintextParser.from_string(
    doc.iloc[0].split("\n", 1)[1],
    Tokenizer("english")
)

summary = summarizer(parser.document, sentences_count=5)

summary

(<Sentence: The 21-year-old singer won the award for best female artist, with Australian Idol runner-up Shannon Noll taking the title of best male at the ceremony.>,
 <Sentence: As well as best female, Goodrem also took home the Pepsi Viewers Choice Award, whilst Green Day bagged the prize for best rock video for American Idiot.>,
 <Sentence: The Black Eyed Peas won awards for best R 'n' B video and sexiest video, both for Hey Mama.>,
 <Sentence: Local singer and songwriter Missy Higgins took the title of breakthrough artist of the year, with Australian Idol winner Guy Sebastian taking the honours for best pop video.>,
 <Sentence: The ceremony was held at the Luna Park fairground in Sydney Harbour and was hosted by the Osbourne family.>)

In [23]:
for s in summary:
    print(wrap(str(s)))

The 21-year-old singer won the award for best female artist, with
Australian Idol runner-up Shannon Noll taking the title of best male
at the ceremony.
As well as best female, Goodrem also took home the Pepsi Viewers
Choice Award, whilst Green Day bagged the prize for best rock video
for American Idiot.
The Black Eyed Peas won awards for best R 'n' B video and sexiest
video, both for Hey Mama.
Local singer and songwriter Missy Higgins took the title of
breakthrough artist of the year, with Australian Idol winner Guy
Sebastian taking the honours for best pop video.
The ceremony was held at the Luna Park fairground in Sydney Harbour
and was hosted by the Osbourne family.


In [24]:
summarizer = LsaSummarizer()

summary = summarizer(parser.document,sentences_count=5)

for s in summary:
    print(wrap(str(s)))


Goodrem, known in both Britain and Australia for her role as Nina
Tucker in TV soap Neighbours, also performed a duet with boyfriend
Brian McFadden.
Other winners included Green Day, voted best group, and the Black Eyed
Peas.
Goodrem, Green Day and the Black Eyed Peas took home two awards each.
As well as best female, Goodrem also took home the Pepsi Viewers
Choice Award, whilst Green Day bagged the prize for best rock video
for American Idiot.
Artists including Carmen Electra, Missy Higgins, Kelly Osbourne, Green
Day, Ja Rule and Natalie Imbruglia gave live performances at the
event.


In [25]:
# https://radimrehurek.com/gensim_3.8.3/summarization/summariser.html
# https://arxiv.org/abs/1602.03606
# Parameters
# text (str) – Given text.
# ratio (float, optional) – Number between 0 and 1 that determines the
#     proportion of the number of sentences of the original text to be
#     chosen for the summary.
# word_count (int or None, optional) – Determines how many words will the
#     output contain. If both parameters are provided, the ratio will be
#     ignored.
# split (bool, optional) – If True, list of sentences will be returned.
#     Otherwise joined strings will bwe returned.
text = doc.iloc[0].split("\n", 1)[1]
sentences = text.split('. ')

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(sentences)
scores = np.array(X.sum(axis=1)).flatten()

# Get top 3 sentences
top_indices = scores.argsort()[-3:][::-1]
summary = ' '.join([sentences[i] for i in top_indices])
print(wrap(summary))


Pop singer Delta Goodrem has scooped one of the top individual prizes
at the first Australian MTV Music Awards.

The 21-year-old singer won
the award for best female artist, with Australian Idol runner-up
Shannon Noll taking the title of best male at the ceremony Local
singer and songwriter Missy Higgins took the title of breakthrough
artist of the year, with Australian Idol winner Guy Sebastian taking
the honours for best pop video Artists including Carmen Electra, Missy
Higgins, Kelly Osbourne, Green Day, Ja Rule and Natalie Imbruglia gave
live performances at the event.


In [26]:
import torch
from transformers import pipeline

# 1. Load the summarization pipeline (this downloads the AI model)
summarizer = pipeline("summarization")

# 2. Get your text (skipping the title, just like your original code)
text = doc.iloc[0].split("\n", 1)[1]

# 3. Generate the summary
# max_length: The summary won't be longer than this many words
# min_length: The summary won't be shorter than this many words
summary = summarizer(text, max_length=130, min_length=30, do_sample=False)

# 4. Print the result using your wrap function
print(wrap(summary[0]['summary_text']))

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
operator torchvision::nms does not exist

![](https://deeplearningcourses.com/notebooks_v3_pxl?sc=N3y7GKxUH16CxQS403JQ2A&n=TextRank+Text+Summarization)